[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/artstock805/pose-image-tool/blob/master/pose_tool.ipynb)

# 원하는 포즈로 이미지 만드는 도구 (SD 1.5 + ControlNet OpenPose)

참조 사진 한 장에서 **사람의 자세(관절)** 만 뽑아, 그 자세를 그대로 유지한 채
프롬프트로 지정한 **다른 인물/장면**을 만드는 노트북입니다.

**실행 순서:** 위에서부터 셀을 하나씩 순서대로 실행하세요.

> ⚠️ 먼저 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU` 로 설정하세요. (무료로 가능)

## 셀 1 — 필요한 라이브러리 설치
이미지 생성(diffusers)과 포즈 추출(controlnet_aux)에 필요한 패키지를 설치합니다.
설치 후 자동 재시작 메시지가 나오면 무시하고 다음 셀로 진행하면 됩니다. (2~3분 소요)

In [ ]:
# 이 셀이 하는 일: 필요한 패키지를 설치한다.
# diffusers를 최신으로 올려 'cached_download' 오류를 해결하고,
# 포즈 추출과 충돌하는 mediapipe는 제거한다. (OpenPose에는 mediapipe가 필요 없음)
!pip -q install -U diffusers transformers accelerate controlnet_aux
!pip -q uninstall -y mediapipe
print("설치 완료! 이제 [런타임 → 세션 다시 시작]을 누른 뒤, 셀 2부터 이어서 실행하세요.")

## 셀 2 — 모델 로드
그림을 그리는 모델(Stable Diffusion 1.5)과, 포즈 조건을 넣어 주는 ControlNet(OpenPose)을 불러옵니다.
처음 실행하면 모델을 내려받느라 몇 분 걸립니다. (한 번만 받으면 됨)

In [ ]:
# 이 셀이 하는 일: 생성 모델(SD 1.5) + 포즈 조건 모델(ControlNet OpenPose)을 GPU에 올린다.
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# 포즈(OpenPose)를 조건으로 받는 ControlNet 불러오기
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose", torch_dtype=torch.float16
)

# 실제 그림을 그리는 Stable Diffusion 1.5 파이프라인 + 위 ControlNet 결합
# (runwayml 저장소가 사라져서 현재 공식 재업로드 주소를 사용)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16
)

# 속도/품질 좋은 스케줄러로 교체하고 GPU로 이동
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # 메모리 절약 (무료 T4에서 안정적)
print("모델 로드 완료!")

## 셀 3 — 참조 사진 올리기 & 포즈(뼈대) 추출
자세를 따올 사람 사진 한 장을 업로드하면, OpenPose가 관절을 찾아 **막대 인형 같은 뼈대 그림**을 만듭니다.
이 뼈대 그림이 다음 셀에서 '자세 조건'으로 쓰입니다.

In [ ]:
# 이 셀이 하는 일: (왼쪽 폴더 패널에 올려둔) 참조 사진을 자동으로 찾아 OpenPose로 관절을 뽑아 뼈대 이미지를 만든다.
# 사진 올리는 법: 왼쪽 📁 폴더 아이콘 → ⬆ 업로드 → 사진 한 장 선택 (파일명은 신경 안 써도 됨)
import glob
from PIL import Image
from controlnet_aux import OpenposeDetector

# /content 폴더에서 이미지 파일을 자동으로 찾기 (jpg/jpeg/png)
candidates = glob.glob("/content/*.jpg") + glob.glob("/content/*.jpeg") + glob.glob("/content/*.png")
ref_name = candidates[0]
print("불러온 사진:", ref_name)

# OpenPose로 관절 추출 → 뼈대(pose) 이미지 생성
reference_image = Image.open(ref_name).convert("RGB")
openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
pose_image = openpose(reference_image)

# 원본과 뼈대 나란히 확인 + 저장
reference_image.save("reference.png")
pose_image.save("pose.png")
display(reference_image.resize((320, 320)))
display(pose_image.resize((320, 320)))
print("포즈 추출 완료! (pose.png 로 저장됨)")

## 셀 4 — 포즈 + 프롬프트로 이미지 생성
위에서 뽑은 자세는 그대로 두고, `prompt`만 바꾸면 원하는 인물/장면이 그 자세로 만들어집니다.
**이 셀의 `prompt`를 바꿔가며 여러 번 실행해 보세요.**

In [ ]:
# 이 셀이 하는 일: 추출한 자세(pose_image) + 프롬프트로 새 이미지를 생성한다.

# ↓↓↓ 여기만 바꿔가며 실험하세요 ↓↓↓
prompt = "an astronaut in a white spacesuit, standing, high quality, detailed"
negative_prompt = "lowres, bad anatomy, blurry, extra limbs, deformed"
seed = 42  # 같은 숫자면 같은 결과, 바꾸면 다른 변형
# ↑↑↑ 여기만 바꿔가며 실험하세요 ↑↑↑

generator = torch.manual_seed(seed)
result = pipe(
    prompt,
    negative_prompt=negative_prompt,
    image=pose_image,          # ← 자세 조건
    num_inference_steps=25,
    generator=generator,
).images[0]

display(result)

## 셀 5 — 결과 저장 & 내려받기
만든 이미지를 파일로 저장하고 내 컴퓨터로 내려받습니다. 이 파일들을 GitHub `samples/` 폴더에 올리면 됩니다.

In [ ]:
# 이 셀이 하는 일: 결과 이미지를 저장하고 내 컴퓨터로 내려받는다.
out_name = "output_01.png"
result.save(out_name)
files.download(out_name)   # 결과 이미지
files.download("pose.png") # 참조 뼈대 이미지
print(f"{out_name} 저장 및 다운로드 완료!")

## 셀 6 — 실험 관찰 기록 (과제 필수 항목)

조건을 바꿔가며 관찰한 내용입니다.

**① 같은 포즈 + 프롬프트만 바꾸기**
- 참조 사진(도시 배경에 서 있는 인물)에서 뽑은 자세를 그대로 두고 프롬프트만 바꿈.
- `우주비행사` → `빨간 한복 입은 여성` → `기사(knight in armor)` 로 바꿨더니,
  **손 위치·서 있는 각도 등 자세는 그대로 유지**되고 옷·인물·배경만 바뀌었다.
- 심지어 뼈대 그림에 함께 잡힌 배경 인물(작은 뼈대)까지 결과 이미지에 그대로 재현되었다.
  → **포즈 조건(ControlNet OpenPose)이 정확히 작동함.**

**② 같은 프롬프트 + 포즈 사진만 바꾸기**
- 서 있는 사진 → 다른 자세의 사진으로 바꾸면, 결과 인물의 자세도 그에 맞게 바뀐다.
- 단, 관절이 겹치거나 옆모습·부분만 보이는 사진은 뼈대 추출이 부정확해서 결과가 어긋나는 경우가 있었다.

**③ 무엇이 결과를 크게 바꾸는가**
- 자세를 크게 바꾸는 것 = **포즈 사진**을 바꿀 때.
- 겉모습·분위기를 크게 바꾸는 것 = **프롬프트**를 바꿀 때.
- 프롬프트에 자세와 모순되는 단어(예: 서 있는 뼈대인데 `sitting`)를 넣으면 결과가 어색해졌다.

**결론:** 자세는 포즈 이미지가, 겉모습·배경은 프롬프트가 결정한다.
정면·전신·관절이 안 겹치는 사진일수록 결과가 자세를 잘 따라온다.